# Бейзлайн и предобработка

Строим бейзлайн с минимальной предобработкой и по шагам проверяем гипотезы из EDA. Каждый шаг сравниваем с предыдущим по RMSE на логарифме цены в кросс-валидации, а результат складываем в общую таблицу в конце

Для проверки берём две модели, потому что предобработка действует на них по-разному:

- Ridge, линейная модель, чувствительна к масштабу, скошенности и выбросам
- LightGBM, деревья, не зависят от масштаба и монотонных преобразований признаков

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")
TARGET = "SalePrice"
RANDOM_STATE = 42

train = pd.read_csv(DATA_DIR / "train.csv")
X = train.drop(columns=[TARGET, "Id"])
y = train[TARGET]

print(f"признаков: {X.shape[1]}, домов: {X.shape[0]}")

признаков: 79, домов: 1460


## 1. Схема валидации и метрика

Метрика соревнования, RMSE между логарифмами предсказанной и реальной цены. Считаем её на `log1p`, как и учим модели, разница с обычным логарифмом на ценах порядка десятков тысяч ничтожна

Валидация, 5 фолдов с 3 повторами и разным перемешиванием. Повторы нужны, чтобы небольшие различия между шагами предобработки не тонули в шуме одного разбиения. Разбиение фиксированное, поэтому все шаги сравниваются на одних и тех же фолдах

In [2]:
def rmse(log_true: pd.Series, log_pred: np.ndarray) -> float:
    """Среднеквадратичная ошибка между двумя массивами логарифмов цены"""
    return float(np.sqrt(np.mean((log_true - log_pred) ** 2)))


def cross_validate(model: Pipeline, X: pd.DataFrame, y: pd.Series, log_target: bool = True) -> tuple[float, float]:
    """Считает RMSE на логарифме цены по повторной кросс-валидации

    Если log_target равен True, модель учится на log1p цены и предсказывает
    сразу логарифм. Иначе учится на сырой цене, а предсказание переводится в
    логарифм с обрезкой отрицательных значений нулём, потому что логарифм
    отрицательного числа не определён. Возвращает среднее и стандартное
    отклонение RMSE по всем фолдам
    """
    folds = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
    log_y = np.log1p(y)
    target = log_y if log_target else y
    scores = []
    for fit_idx, valid_idx in folds.split(X):
        fitted = clone(model).fit(X.iloc[fit_idx], target.iloc[fit_idx])
        prediction = fitted.predict(X.iloc[valid_idx])
        if not log_target:
            prediction = np.log1p(np.clip(prediction, 0, None))
        scores.append(rmse(log_y.iloc[valid_idx], prediction))
    return float(np.mean(scores)), float(np.std(scores))

## 2. Бейзлайн

Минимальная предобработка без единой идеи из EDA:

- числовые пропуски заполняем медианой
- категориальные пропуски заполняем самым частым значением, то есть NA в `PoolQC` и `GarageType` для нас пока обычные потерянные значения
- категории кодируем one-hot, числа для Ridge масштабируем
- `MSSubClass` остаётся числом, цена не логарифмируется

Ещё берём тривиальную модель, которая всегда предсказывает медиану цены. Любая настоящая модель обязана её обойти

In [3]:
def build_preprocessor(scale: bool) -> ColumnTransformer:
    """Минимальная предобработка: медиана для чисел, мода и one-hot для категорий

    scale включает стандартизацию чисел, она нужна линейным моделям
    """
    numeric_steps = [SimpleImputer(strategy="median")] + ([StandardScaler()] if scale else [])
    return ColumnTransformer(
        [
            ("numeric", make_pipeline(*numeric_steps), make_column_selector(dtype_include="number")),
            (
                "categorical",
                make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")),
                make_column_selector(dtype_exclude="number"),
            ),
        ]
    )


def build_models() -> dict[str, Pipeline]:
    """Собирает пайплайны Ridge и LightGBM с одинаковой минимальной предобработкой"""
    return {
        "Ridge": make_pipeline(build_preprocessor(scale=True), Ridge(alpha=10)),
        "LightGBM": make_pipeline(
            build_preprocessor(scale=False), LGBMRegressor(random_state=RANDOM_STATE, verbose=-1)
        ),
    }


results = []


def evaluate_models(experiment: str, X: pd.DataFrame, y: pd.Series, log_target: bool = True) -> pd.DataFrame:
    """Оценивает обе модели на данных X, y и записывает результат в общий список results

    Перед таблицей печатает название эксперимента
    """
    print(f"{experiment}: RMSE на логарифме цены по кросс-валидации")
    rows = []
    for model_name, model in build_models().items():
        rmse, std = cross_validate(model, X, y, log_target)
        rows.append({"experiment": experiment, "model": model_name, "rmse": rmse, "std": std})
    results.extend(rows)
    return pd.DataFrame(rows).set_index("model")[["rmse", "std"]]

In [4]:
median_baseline = make_pipeline(DummyRegressor(strategy="median"))
dummy_rmse, dummy_std = cross_validate(median_baseline, X, y, log_target=False)
print(f"медиана цены: RMSE {dummy_rmse:.4f}, std {dummy_std:.4f}")


evaluate_models("1. Бейзлайн, сырая цена", X, y, log_target=False)

медиана цены: RMSE 0.3998, std 0.0177
1. Бейзлайн, сырая цена: RMSE на логарифме цены по кросс-валидации


,rmse,std
model,,
Ridge,0.153805,0.015969
LightGBM,0.135922,0.011898


## 3. Гипотеза 1: обучение на логарифме цены

Первая гипотеза из EDA, скошенность цены падает с 1.88 до 0.12, поэтому модели должно быть проще учиться на `log1p(SalePrice)`. Меняем только это, всё остальное как в бейзлайне

In [5]:
evaluate_models("2. Обучение на log1p цены", X, y, log_target=True)

2. Обучение на log1p цены: RMSE на логарифме цены по кросс-валидации


,rmse,std
model,,
Ridge,0.143333,0.031774
LightGBM,0.132933,0.011904


In [6]:
def show_results() -> pd.DataFrame:
    """Собирает результаты всех экспериментов в таблицу RMSE по моделям"""
    table = pd.DataFrame(results).pivot(index="experiment", columns="model", values="rmse")
    return table[["Ridge", "LightGBM"]].round(4)


show_results()

model,Ridge,LightGBM
experiment,,
"1. Бейзлайн, сырая цена",0.1538,0.1359
2. Обучение на log1p цены,0.1433,0.1329


### Выводы по бейзлайну и гипотезе 1

- Медиана цены даёт RMSE 0.3998, это нижняя планка для сравнения
- Бейзлайн с минимальной предобработкой уже сильно лучше: Ridge 0.1538, LightGBM 0.1359, разброс между фолдами 0.016 и 0.012
- Гипотеза 1 подтвердилась: обучение на `log1p` цены снижает RMSE у Ridge с 0.1538 до 0.1433 и у LightGBM с 0.1359 до 0.1329. Линейной модели логарифм помогает заметно сильнее, деревьям почти не важен, что ожидаемо для модели, которая делит по порогам
- У Ridge на логарифме вырос разброс между фолдами с 0.016 до 0.032, то есть в каких-то фолдах модель ошибается сильно. Похоже на выбросы, проверим это в гипотезе 4
- Дальше все эксперименты идут с обучением на логарифме цены